## Score: 0.75619

## 1. Установка необходимых библиотек

In [ ]:
!pip install --upgrade featuretools >> None

## 2. Импорт библиотек и настройка

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from catboost import CatBoostRegressor, Pool

import warnings
warnings.filterwarnings("ignore")

## 3. Загрузка данных

In [ ]:
transaction_file = '/kaggle/input/dataset-generated/dataset_generated_with_cats.csv'
dataset_generated_with_cats = pd.read_csv(transaction_file)

In [ ]:
target_file = '/kaggle/input/alfa-challenge/train.pa'
df_target = pd.read_parquet(target_file)

In [ ]:
print("Данные успешно загружены.")

## 4. Подготовка данных 

In [ ]:
target = df_target[['client_num', 'target']]

data_for_model = dataset_generated_with_cats.merge(target, on='client_num', how='inner')
print("Признаки и целевая переменная объединены для модели.")

cat_features = data_for_model.select_dtypes(include=['object', 'category']).columns.tolist()

X = data_for_model.drop(['client_num', 'target'], axis=1)
y = data_for_model['target']

X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print("Данные разделены на обучающую и валидационную выборки.")

## 5. Расчет весов классов

In [ ]:
unique_classes = np.sort(y.unique())
class_counts = y_train.value_counts()
total_samples = len(y_train)
class_weights = {c: total_samples / (len(unique_classes) * class_counts[c]) for c in unique_classes}
print("Веса классов рассчитаны.")

def map_class_weights(y_labels, class_weights):
    return y_labels.map(class_weights).values

class WMAEMetric:
    def get_final_error(self, error, weight):
        return error / weight

    def is_max_optimal(self):
        return False

    def evaluate(self, approxes, target, weight):
        approx = approxes[0]
        target = np.array(target)
        weight = np.ones_like(target) if weight is None else np.array(weight)
        error = np.sum(weight * np.abs(target - approx))
        return error, np.sum(weight)
print("Кастомная метрика WMAE определена.")

weights_train = map_class_weights(y_train, class_weights)
weights_valid = map_class_weights(y_valid, class_weights)
print("Веса для выборок рассчитаны.")

## 6. Создание пулов данных для CatBoost

In [ ]:
train_pool = Pool(
    data=X_train, 
    label=y_train, 
    weight=weights_train, 
    cat_features=cat_features
)
valid_pool = Pool(
    data=X_valid, 
    label=y_valid, 
    weight=weights_valid, 
    cat_features=cat_features
)
print("Данные подготовлены для CatBoost.")

## 7. Настройка и обучение модели

In [ ]:
params = {
    'iterations': 1_000_000,
    'depth': 11,
    'l2_leaf_reg': 2,
    'colsample_bylevel': 0.4,
    'boosting_type': 'Plain',
    'bootstrap_type': 'MVS',
    'eval_metric': WMAEMetric(),
    'loss_function': 'MAE',
    'random_seed': 42,
    'verbose': 1,
    'early_stopping_rounds': 300,
    'use_best_model': True
}
print("Параметры модели настроены.")

model = CatBoostRegressor(**params)
model.fit(
    train_pool,
    eval_set=valid_pool,
    verbose=True
)
print("Модель обучена.")

y_valid_pred = model.predict(X_valid)
wmae = np.sum(weights_valid * np.abs(y_valid - y_valid_pred)) / np.sum(weights_valid)
print('Validation WMAE:', wmae)

## 8. Предсказание на тестовых данных

In [ ]:
all_client_nums = dataset_generated_with_cats['client_num']
train_client_nums = df_target['client_num']
test_client_nums = all_client_nums[~all_client_nums.isin(train_client_nums)]

test_features = dataset_generated_with_cats[dataset_generated_with_cats['client_num'].isin(test_client_nums)]

X_test = test_features.drop(['client_num'], axis=1)

test_pool = Pool(
    data=X_test,
    cat_features=cat_features
)

test_predictions = model.predict(test_pool)
test_predictions_rounded = np.round(test_predictions).astype(int)

## 9. Результат

In [ ]:
submission = pd.DataFrame({
    'client_num': test_client_nums,
    'target': test_predictions_rounded
})

submission.to_csv('test_with_round.csv', index=False)
print("Предсказания сохранены в файл 'test_with_round.csv'.")

submission.head(5)